In [2]:
import pandas as pd

df = pd.read_csv('data.csv')

In [3]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)


2026-01-18 18:09:42.234204: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = 224
batch_size=16

split_id = int(.8*df.shape[0])
train_X, val_X = df.paths[:split_id], df.paths[split_id:]
train_y, val_y = df.age[:split_id], df.age[split_id:]

def image_feeder(path, age):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    age = tf.cast(age, tf.float32)
    return img, age

train_ds = tf.data.Dataset.from_tensor_slices((train_X, train_y))
train_ds = (train_ds
    .shuffle(buffer_size=min(len(train_X), 5000), reshuffle_each_iteration=True)
    .map(image_feeder, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(2)
    .ignore_errors()
)

val_ds = tf.data.Dataset.from_tensor_slices((val_X, val_y))
val_ds = (val_ds
    .map(image_feeder, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(2)
    .ignore_errors()
)


I0000 00:00:1768777812.671400   19266 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9558 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:0a:00.0, compute capability: 8.9


In [5]:
IMG_SHAPE = (224, 224, 3)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # ok even though already in feeder; see note below
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1)(x)  # regression

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)


In [6]:
import math
steps_per_epoch = math.ceil(len(train_X) / batch_size)
val_steps = math.ceil(len(val_X) / batch_size)

train_ds = train_ds.repeat()
val_ds = val_ds.repeat()


In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
)


Epoch 1/10


2026-01-18 18:10:28.713904: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91800


23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1166s 50ms/step - loss: 1327.3087 - mae: 11.5296 - val_loss: 1101.9264 - val_mae: 10.4853
Epoch 2/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1070s 46ms/step - loss: 1302.1775 - mae: 10.9684 - val_loss: 1100.3114 - val_mae: 10.4818
Epoch 3/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1074s 47ms/step - loss: 1301.8907 - mae: 10.9552 - val_loss: 1099.8331 - val_mae: 10.4531
Epoch 4/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1051s 46ms/step - loss: 1301.0411 - mae: 10.9435 - val_loss: 1099.3844 - val_mae: 10.4344
Epoch 5/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1171s 51ms/step - loss: 1300.6050 - mae: 10.9327 - val_loss: 1098.7886 - val_mae: 10.4260
Epoch 6/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1025s 44ms/step - loss: 1300.3297 - mae: 10.9236 - val_loss: 1098.2439 - val_mae: 10.4218
Epoch 7/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1260s 55ms/step - loss: 1300.2972 - mae: 10.9216 - val_loss: 1098.2524 - val_mae: 10.4011
Epoch 8/10
23037/23037 ━━━━━━━━━━━━━━━━━━━━ 1079s 47ms/step - loss: 1299.

In [2]:


model.save("/mnt/c/Users/Walker/Documents/website_1/saved_models/age_mobilenetv2.keras")
#model.save('age_model_1_no_croping.keras')

NameError: name 'model' is not defined

In [1]:
import tensorflow as tf

IMG_SIZE = 224

def load_one_image(path):
    b = tf.io.read_file(path)
    x = tf.image.decode_jpeg(b, channels=3)
    x = tf.image.resize(x, [IMG_SIZE, IMG_SIZE])
    x = tf.cast(x, tf.float32)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    return tf.expand_dims(x, 0)  # (1,224,224,3)

my_path = "/mnt/c/Users/Walker/Downloads/linkedin_face.jpg"  # change this
x = load_one_image(my_path)
pred_age = float(model.predict(x, verbose=0)[0][0])
print("Predicted age:", pred_age)


2026-01-18 21:18:00.992593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1768789146.474603  112024 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9558 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:0a:00.0, compute capability: 8.9


NameError: name 'model' is not defined